In [1]:
import pandas as pd
import numpy as np
import json
from scipy.stats import poisson
from xgboost import XGBRegressor
from collections import defaultdict

In [2]:
home_model = XGBRegressor()
home_model.load_model("../models/home_goals_model.json")

away_model = XGBRegressor()
away_model.load_model("../models/away_goals_model.json")

with open("../models/dixon_coles_params.json") as f:
    dc_params = json.load(f)

RHO = dc_params["rho"]
print(f"Loaded rho: {RHO}")

Loaded rho: -0.29


In [3]:
team_vectors = pd.read_csv("../data/processed/team_vectors.csv")
shootouts = pd.read_csv("../data/raw/kaggle/shootouts.csv")

print(f"Teams: {len(team_vectors)}")
print(f"Shootout records: {len(shootouts)}")
print(team_vectors.columns.tolist())

Teams: 48
Shootout records: 678
['team', 'atk_overall', 'atk_shooting', 'atk_pace', 'atk_dribbling', 'mid_overall', 'mid_passing', 'mid_physic', 'def_overall', 'def_defending', 'def_physic', 'gk_overall', 'top11_overall', 'squad_depth', 'avg_caps', 'atk_xG_per90', 'atk_xA_per90']


In [4]:
def scale(val, in_min, in_max, out_min, out_max):
    return out_min + (val - in_min) / (in_max - in_min) * (out_max - out_min)

def team_vector_to_elo(row):
    score = (
        0.4 * row['top11_overall'] +
        0.2 * row['atk_overall'] +
        0.2 * row['def_overall'] +
        0.1 * row['mid_overall'] +
        0.1 * row['gk_overall']
    )
    return scale(score, in_min=70, in_max=90, out_min=1400, out_max=2100)

# Build elo dict
team_elo = {}
for _, row in team_vectors.iterrows():
    base_elo = team_vector_to_elo(row)
    
    # Form nudge for teams with Understat xG data
    if pd.notna(row['atk_xG_per90']):
        # league average xG/90 for attackers is roughly 0.35
        # nudge up to +/- 50 Elo points based on deviation from average
        xg_nudge = (row['atk_xG_per90'] - 0.35) * 100
        xg_nudge = np.clip(xg_nudge, -50, 50)
    else:
        xg_nudge = 0
    
    team_elo[row['team']] = round(base_elo + xg_nudge, 2)

# Sanity check — print top 10
sorted_elo = sorted(team_elo.items(), key=lambda x: x[1], reverse=True)
print("Top 10 by synthetic Elo:")
for team, elo in sorted_elo[:10]:
    print(f"  {team}: {elo}")

print("\nBottom 5:")
for team, elo in sorted_elo[-5:]:
    print(f"  {team}: {elo}")

Top 10 by synthetic Elo:
  France: 1917.22
  Spain: 1916.25
  Germany: 1885.82
  England: 1870.48
  Brazil: 1864.3
  Portugal: 1852.55
  Argentina: 1830.41
  Netherlands: 1821.84
  Belgium: 1763.23
  Norway: 1732.84

Bottom 5:
  Curaçao: 1479.53
  Saudi Arabia: 1415.85
  New Zealand: 1387.66
  Australia: 1384.06
  Cape Verde: 1336.81


In [5]:
def dc_tau(home_goals, away_goals, lambda_home, lambda_away, rho):
    if home_goals == 0 and away_goals == 0:
        return max(1 - lambda_home * lambda_away * rho, 1e-6)
    elif home_goals == 1 and away_goals == 0:
        return max(1 + lambda_away * rho, 1e-6)
    elif home_goals == 0 and away_goals == 1:
        return max(1 + lambda_home * rho, 1e-6)
    elif home_goals == 1 and away_goals == 1:
        return max(1 - rho, 1e-6)
    else:
        return 1.0

def scoreline_probs(lambda_home, lambda_away, rho, max_goals=8):
    grid = np.zeros((max_goals + 1, max_goals + 1))
    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            grid[h, a] = (
                poisson.pmf(h, lambda_home) *
                poisson.pmf(a, lambda_away) *
                dc_tau(h, a, lambda_home, lambda_away, rho)
            )
    grid /= grid.sum()
    return grid

def sample_scoreline(lambda_home, lambda_away, rho):
    grid = scoreline_probs(lambda_home, lambda_away, rho)
    probs_flat = grid.flatten()
    idx = np.random.choice(len(probs_flat), p=probs_flat)
    h = idx // grid.shape[1]
    a = idx % grid.shape[1]
    return h, a

In [43]:
def build_penalty_lookup(shootouts_df):
    lookup = {}
    for _, row in shootouts_df.iterrows():
        winner = row['winner']
        home = row['home_team']
        away = row['away_team']
        loser = away if winner == home else home
        lookup[winner] = lookup.get(winner, {'w': 0, 'l': 0})
        lookup[loser]  = lookup.get(loser,  {'w': 0, 'l': 0})
        lookup[winner]['w'] += 1
        lookup[loser]['l']  += 1
    
    win_rates = {}
    for team, record in lookup.items():
        total = record['w'] + record['l']
        win_rates[team] = max(record['w'] / total, 0.1)
    return win_rates

penalty_lookup = build_penalty_lookup(shootouts)

# Verify
for team in ['Curaçao', 'Austria', 'Jordan']:
    print(f"{team}: {penalty_lookup.get(team, 0.5)}")

def simulate_penalty(team_a, team_b):
    """Returns winning team. Falls back to 50/50 if no record."""
    rate_a = penalty_lookup.get(team_a, 0.5)
    rate_b = penalty_lookup.get(team_b, 0.5)
    # Normalise so they sum to 1
    total = rate_a + rate_b
    prob_a = rate_a / total
    return team_a if np.random.random() < prob_a else team_b

Curaçao: 0.1
Austria: 0.1
Jordan: 0.1


In [17]:
FEATURE_COLS = ["home_elo", "away_elo", "elo_diff", "is_worldcup"]

def predict_lambdas(home_team, away_team, is_wc=1):
    h_elo = team_elo[home_team]
    a_elo = team_elo[away_team]
    features = pd.DataFrame([{
        "home_elo": h_elo,
        "away_elo": a_elo,
        "elo_diff": h_elo - a_elo,
        "is_worldcup": is_wc
    }])
    lam_h = float(np.clip(home_model.predict(features)[0], 0.1, 2.5))
    lam_a = float(np.clip(away_model.predict(features)[0], 0.1, 2.5))
    return lam_h, lam_a

def simulate_group_match(home_team, away_team):
    """Returns (home_goals, away_goals) — draws allowed."""
    lam_h, lam_a = predict_lambdas(home_team, away_team)
    return sample_scoreline(lam_h, lam_a, RHO)

def simulate_knockout_match(team_a, team_b):
    """Returns (winner, team_a_goals, team_b_goals, method)."""
    lam_a, lam_b = predict_lambdas(team_a, team_b)
    h, a = sample_scoreline(lam_a, lam_b, RHO)
    
    if h != a:
        winner = team_a if h > a else team_b
        return (winner, h, a, 'normal')
    
    # Extra time
    et_h, et_a = sample_scoreline(lam_a * 0.33, lam_b * 0.33, RHO)
    h += et_h
    a += et_a
    
    if h != a:
        winner = team_a if h > a else team_b
        return (winner, h, a, 'et')
    
    # Penalties
    winner = simulate_penalty(team_a, team_b)
    return (winner, h, a, 'pens')

In [18]:
# Test a few matches
for _ in range(5):
    result = simulate_knockout_match("France", "Morocco")
    print(result)

('France', 2, 1, 'et')
('France', 1, 0, 'normal')
('France', 1, 0, 'normal')
('France', 2, 2, 'pens')
('France', 3, 1, 'normal')


In [19]:
lam_h, lam_a = predict_lambdas("France", "Morocco")
print(f"France lambda: {lam_h:.3f}")
print(f"Morocco lambda: {lam_a:.3f}")

France lambda: 2.422
Morocco lambda: 0.574


In [20]:
matchups = [
    ("France", "Brazil"),
    ("Spain", "Germany"),
    ("Argentina", "England"),
    ("Brazil", "Saudi Arabia"),
    ("United States", "Mexico"),
    ("Norway", "Australia"),
]

for home, away in matchups:
    lam_h, lam_a = predict_lambdas(home, away)
    print(f"{home} vs {away}: {lam_h:.3f} - {lam_a:.3f}")

France vs Brazil: 1.531 - 1.087
Spain vs Germany: 1.552 - 1.126
Argentina vs England: 1.197 - 1.447
Brazil vs Saudi Arabia: 2.500 - 0.423
United States vs Mexico: 1.462 - 1.014
Norway vs Australia: 2.500 - 0.411


In [23]:
# WC 2026 Groups — exact draw
GROUPS = {
    'A': ['Mexico', 'South Korea', 'Czech Republic', 'South Africa'],
    'B': ['Canada', 'Switzerland', 'Qatar', 'Bosnia and Herzegovina'],
    'C': ['Brazil', 'Morocco', 'Scotland', 'Haiti'],
    'D': ['United States', 'Australia', 'Turkey', 'Paraguay'],
    'E': ['Germany', 'Ivory Coast', 'Ecuador', 'Curaçao'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Uruguay', 'Saudi Arabia', 'Cape Verde'],
    'I': ['France', 'Senegal', 'Norway', 'Iraq'],
    'J': ['Argentina', 'Austria', 'Algeria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

# Fixtures per group — matchday structure from Article 12.4
# (1v2, 3v4), (1v3, 4v2), (4v1, 2v3)
def get_group_fixtures(teams):
    t1, t2, t3, t4 = teams
    return [
        (t1, t2), (t3, t4),  # matchday 1
        (t1, t3), (t4, t2),  # matchday 2
        (t4, t1), (t2, t3),  # matchday 3
    ]

# Verify all teams exist in team_elo
missing = []
for group, teams in GROUPS.items():
    for team in teams:
        if team not in team_elo:
            missing.append(f"Group {group}: {team}")

if missing:
    print("MISSING FROM team_elo:")
    for m in missing:
        print(f"  {m}")
else:
    print("All 48 teams found in team_elo")

All 48 teams found in team_elo


In [24]:
def simulate_group(group_name, teams):
    """
    Simulate all 6 group matches, return ranked list of 4 teams with stats.
    Each team dict: {team, pts, gd, gf, ga, h2h_results}
    """
    fixtures = get_group_fixtures(teams)
    
    # Initialize table
    table = {team: {
        'team': team,
        'group': group_name,
        'pts': 0,
        'gf': 0,
        'ga': 0,
        'gd': 0,
        'w': 0, 'd': 0, 'l': 0,
    } for team in teams}
    
    # Store all match results for H2H lookup
    results = {}  # (home, away) -> (hg, ag)
    
    for home, away in fixtures:
        hg, ag = simulate_group_match(home, away)
        results[(home, away)] = (hg, ag)
        
        # Update goals
        table[home]['gf'] += hg
        table[home]['ga'] += ag
        table[away]['gf'] += ag
        table[away]['ga'] += hg
        
        # Update points
        if hg > ag:
            table[home]['pts'] += 3
            table[home]['w'] += 1
            table[away]['l'] += 1
        elif ag > hg:
            table[away]['pts'] += 3
            table[away]['w'] += 1
            table[home]['l'] += 1
        else:
            table[home]['pts'] += 1
            table[away]['pts'] += 1
            table[home]['d'] += 1
            table[away]['d'] += 1
    
    # Update GD
    for team in teams:
        table[team]['gd'] = table[team]['gf'] - table[team]['ga']
    
    # Sort with tiebreakers
    ranked = rank_group(list(table.values()), results)
    return ranked, results


def get_h2h_stats(tied_teams, results):
    """Get H2H points, GD, GF among a subset of tied teams."""
    h2h = {t: {'pts': 0, 'gd': 0, 'gf': 0} for t in tied_teams}
    tied_set = set(tied_teams)
    
    for t1 in tied_teams:
        for t2 in tied_teams:
            if t1 == t2:
                continue
            # Check both directions
            if (t1, t2) in results:
                hg, ag = results[(t1, t2)]
                h2h[t1]['gf'] += hg
                h2h[t2]['gf'] += ag
                h2h[t1]['gd'] += hg - ag
                h2h[t2]['gd'] += ag - hg
                if hg > ag:
                    h2h[t1]['pts'] += 3
                elif ag > hg:
                    h2h[t2]['pts'] += 3
                else:
                    h2h[t1]['pts'] += 1
                    h2h[t2]['pts'] += 1
            elif (t2, t1) in results:
                hg, ag = results[(t2, t1)]
                h2h[t2]['gf'] += hg
                h2h[t1]['gf'] += ag
                h2h[t2]['gd'] += hg - ag
                h2h[t1]['gd'] += ag - hg
                if hg > ag:
                    h2h[t2]['pts'] += 3
                elif ag > hg:
                    h2h[t1]['pts'] += 3
                else:
                    h2h[t1]['pts'] += 1
                    h2h[t2]['pts'] += 1
    return h2h


def rank_group(team_rows, results):
    """
    Rank 4 teams using FIFA tiebreaker chain:
    1. Points
    2. H2H points (among tied teams)
    3. H2H GD
    4. H2H GF
    5. Overall GD
    6. Overall GF
    7. Synthetic Elo (proxy for FIFA ranking)
    """
    def sort_key_overall(t):
        return (t['pts'], t['gd'], t['gf'], team_elo.get(t['team'], 1500))
    
    # First pass — sort by points
    team_rows.sort(key=lambda t: t['pts'], reverse=True)
    
    # Find groups of tied teams and apply H2H within each group
    final_ranked = []
    i = 0
    while i < len(team_rows):
        # Find all teams tied on points with team_rows[i]
        tied = [team_rows[i]]
        j = i + 1
        while j < len(team_rows) and team_rows[j]['pts'] == team_rows[i]['pts']:
            tied.append(team_rows[j])
            j += 1
        
        if len(tied) == 1:
            final_ranked.append(tied[0])
        else:
            # Apply H2H tiebreakers within tied group
            tied_names = [t['team'] for t in tied]
            h2h = get_h2h_stats(tied_names, results)
            
            tied.sort(key=lambda t: (
                h2h[t['team']]['pts'],
                h2h[t['team']]['gd'],
                h2h[t['team']]['gf'],
                t['gd'],
                t['gf'],
                team_elo.get(t['team'], 1500)
            ), reverse=True)
            final_ranked.extend(tied)
        
        i = j
    
    return final_ranked

In [30]:
# Simulate Group I (France, Senegal, Norway, Iraq) 5 times
for i in range(5):
    ranked, _ = simulate_group('C', GROUPS['C'])
    print(f"Run {i+1}: " + " | ".join([f"{r['team']} ({r['pts']}pts)" for r in ranked]))

Run 1: Haiti (6pts) | Morocco (6pts) | Brazil (3pts) | Scotland (3pts)
Run 2: Brazil (9pts) | Morocco (4pts) | Scotland (3pts) | Haiti (1pts)
Run 3: Morocco (7pts) | Brazil (5pts) | Scotland (3pts) | Haiti (1pts)
Run 4: Brazil (7pts) | Morocco (5pts) | Scotland (4pts) | Haiti (0pts)
Run 5: Morocco (7pts) | Scotland (4pts) | Brazil (4pts) | Haiti (1pts)


In [31]:
# Annexe C as a list of 495 rows
# Each row: (option_number, {slot: group_letter})
# Columns: 1A, 1B, 1D, 1E, 1G, 1I, 1K, 1L

ANNEXE_C_RAW = """1 3E 3J 3I 3F 3H 3G 3L 3K
2 3H 3G 3I 3D 3J 3F 3L 3K
3 3E 3J 3I 3D 3H 3G 3L 3K
4 3E 3J 3I 3D 3H 3F 3L 3K
5 3E 3G 3I 3D 3J 3F 3L 3K
6 3E 3G 3J 3D 3H 3F 3L 3K
7 3E 3G 3I 3D 3H 3F 3L 3K
8 3E 3G 3J 3D 3H 3F 3L 3I
9 3E 3G 3J 3D 3H 3F 3I 3K
10 3H 3G 3I 3C 3J 3F 3L 3K
11 3E 3J 3I 3C 3H 3G 3L 3K
12 3E 3J 3I 3C 3H 3F 3L 3K
13 3E 3G 3I 3C 3J 3F 3L 3K
14 3E 3G 3J 3C 3H 3F 3L 3K
15 3E 3G 3I 3C 3H 3F 3L 3K
16 3E 3G 3J 3C 3H 3F 3L 3I
17 3E 3G 3J 3C 3H 3F 3I 3K
18 3H 3G 3I 3C 3J 3D 3L 3K
19 3C 3J 3I 3D 3H 3F 3L 3K
20 3C 3G 3I 3D 3J 3F 3L 3K
21 3C 3G 3J 3D 3H 3F 3L 3K
22 3C 3G 3I 3D 3H 3F 3L 3K
23 3C 3G 3J 3D 3H 3F 3L 3I
24 3C 3G 3J 3D 3H 3F 3I 3K
25 3E 3J 3I 3C 3H 3D 3L 3K
26 3E 3G 3I 3C 3J 3D 3L 3K
27 3E 3G 3J 3C 3H 3D 3L 3K
28 3E 3G 3I 3C 3H 3D 3L 3K
29 3E 3G 3J 3C 3H 3D 3L 3I
30 3E 3G 3J 3C 3H 3D 3I 3K
31 3C 3J 3E 3D 3I 3F 3L 3K
32 3C 3J 3E 3D 3H 3F 3L 3K
33 3C 3E 3I 3D 3H 3F 3L 3K
34 3C 3J 3E 3D 3H 3F 3L 3I
35 3C 3J 3E 3D 3H 3F 3I 3K
36 3C 3G 3E 3D 3J 3F 3L 3K
37 3C 3G 3E 3D 3I 3F 3L 3K
38 3C 3G 3E 3D 3J 3F 3L 3I
39 3C 3G 3E 3D 3J 3F 3I 3K
40 3C 3G 3E 3D 3H 3F 3L 3K
41 3C 3G 3J 3D 3H 3F 3L 3E
42 3C 3G 3J 3D 3H 3F 3E 3K
43 3C 3G 3E 3D 3H 3F 3L 3I
44 3C 3G 3E 3D 3H 3F 3I 3K
45 3C 3G 3J 3D 3H 3F 3E 3I
46 3H 3J 3B 3F 3I 3G 3L 3K
47 3E 3J 3I 3B 3H 3G 3L 3K
48 3E 3J 3B 3F 3I 3H 3L 3K
49 3E 3J 3B 3F 3I 3G 3L 3K
50 3E 3J 3B 3F 3H 3G 3L 3K
51 3E 3G 3B 3F 3I 3H 3L 3K
52 3E 3J 3B 3F 3H 3G 3L 3I
53 3E 3J 3B 3F 3H 3G 3I 3K
54 3H 3J 3B 3D 3I 3G 3L 3K
55 3H 3J 3B 3D 3I 3F 3L 3K
56 3I 3G 3B 3D 3J 3F 3L 3K
57 3H 3G 3B 3D 3J 3F 3L 3K
58 3H 3G 3B 3D 3I 3F 3L 3K
59 3H 3G 3B 3D 3J 3F 3L 3I
60 3H 3G 3B 3D 3J 3F 3I 3K
61 3E 3J 3B 3D 3I 3H 3L 3K
62 3E 3J 3B 3D 3I 3G 3L 3K
63 3E 3J 3B 3D 3H 3G 3L 3K
64 3E 3G 3B 3D 3I 3H 3L 3K
65 3E 3J 3B 3D 3H 3G 3L 3I
66 3E 3J 3B 3D 3H 3G 3I 3K
67 3E 3J 3B 3D 3I 3F 3L 3K
68 3E 3J 3B 3D 3H 3F 3L 3K
69 3E 3I 3B 3D 3H 3F 3L 3K
70 3E 3J 3B 3D 3H 3F 3L 3I
71 3E 3J 3B 3D 3H 3F 3I 3K
72 3E 3G 3B 3D 3J 3F 3L 3K
73 3E 3G 3B 3D 3I 3F 3L 3K
74 3E 3G 3B 3D 3J 3F 3L 3I
75 3E 3G 3B 3D 3J 3F 3I 3K
76 3E 3G 3B 3D 3H 3F 3L 3K
77 3H 3G 3B 3D 3J 3F 3L 3E
78 3H 3G 3B 3D 3J 3F 3E 3K
79 3E 3G 3B 3D 3H 3F 3L 3I
80 3E 3G 3B 3D 3H 3F 3I 3K
81 3H 3G 3B 3D 3J 3F 3E 3I
82 3H 3J 3B 3C 3I 3G 3L 3K
83 3H 3J 3B 3C 3I 3F 3L 3K
84 3I 3G 3B 3C 3J 3F 3L 3K
85 3H 3G 3B 3C 3J 3F 3L 3K
86 3H 3G 3B 3C 3I 3F 3L 3K
87 3H 3G 3B 3C 3J 3F 3L 3I
88 3H 3G 3B 3C 3J 3F 3I 3K
89 3E 3J 3B 3C 3I 3H 3L 3K
90 3E 3J 3B 3C 3I 3G 3L 3K
91 3E 3J 3B 3C 3H 3G 3L 3K
92 3E 3G 3B 3C 3I 3H 3L 3K
93 3E 3J 3B 3C 3H 3G 3L 3I
94 3E 3J 3B 3C 3H 3G 3I 3K
95 3E 3J 3B 3C 3I 3F 3L 3K
96 3E 3J 3B 3C 3H 3F 3L 3K
97 3E 3I 3B 3C 3H 3F 3L 3K
98 3E 3J 3B 3C 3H 3F 3L 3I
99 3E 3J 3B 3C 3H 3F 3I 3K
100 3E 3G 3B 3C 3J 3F 3L 3K
101 3E 3G 3B 3C 3I 3F 3L 3K
102 3E 3G 3B 3C 3J 3F 3L 3I
103 3E 3G 3B 3C 3J 3F 3I 3K
104 3E 3G 3B 3C 3H 3F 3L 3K
105 3H 3G 3B 3C 3J 3F 3L 3E
106 3H 3G 3B 3C 3J 3F 3E 3K
107 3E 3G 3B 3C 3H 3F 3L 3I
108 3E 3G 3B 3C 3H 3F 3I 3K
109 3H 3G 3B 3C 3J 3F 3E 3I
110 3H 3J 3B 3C 3I 3D 3L 3K
111 3I 3G 3B 3C 3J 3D 3L 3K
112 3H 3G 3B 3C 3J 3D 3L 3K
113 3H 3G 3B 3C 3I 3D 3L 3K
114 3H 3G 3B 3C 3J 3D 3L 3I
115 3H 3G 3B 3C 3J 3D 3I 3K
116 3C 3J 3B 3D 3I 3F 3L 3K
117 3C 3J 3B 3D 3H 3F 3L 3K
118 3C 3I 3B 3D 3H 3F 3L 3K
119 3C 3J 3B 3D 3H 3F 3L 3I
120 3C 3J 3B 3D 3H 3F 3I 3K
121 3C 3G 3B 3D 3J 3F 3L 3K
122 3C 3G 3B 3D 3I 3F 3L 3K
123 3C 3G 3B 3D 3J 3F 3L 3I
124 3C 3G 3B 3D 3J 3F 3I 3K
125 3C 3G 3B 3D 3H 3F 3L 3K
126 3C 3G 3B 3D 3H 3F 3L 3J
127 3H 3G 3B 3C 3J 3F 3D 3K
128 3C 3G 3B 3D 3H 3F 3L 3I
129 3C 3G 3B 3D 3H 3F 3I 3K
130 3H 3G 3B 3C 3J 3F 3D 3I
131 3E 3J 3B 3C 3I 3D 3L 3K
132 3E 3J 3B 3C 3H 3D 3L 3K
133 3E 3I 3B 3C 3H 3D 3L 3K
134 3E 3J 3B 3C 3H 3D 3L 3I
135 3E 3J 3B 3C 3H 3D 3I 3K
136 3E 3G 3B 3C 3J 3D 3L 3K
137 3E 3G 3B 3C 3I 3D 3L 3K
138 3E 3G 3B 3C 3J 3D 3L 3I
139 3E 3G 3B 3C 3J 3D 3I 3K
140 3E 3G 3B 3C 3H 3D 3L 3K
141 3H 3G 3B 3C 3J 3D 3L 3E
142 3H 3G 3B 3C 3J 3D 3E 3K
143 3E 3G 3B 3C 3H 3D 3L 3I
144 3E 3G 3B 3C 3H 3D 3I 3K
145 3H 3G 3B 3C 3J 3D 3E 3I
146 3C 3J 3B 3D 3E 3F 3L 3K
147 3C 3E 3B 3D 3I 3F 3L 3K
148 3C 3J 3B 3D 3E 3F 3L 3I
149 3C 3J 3B 3D 3E 3F 3I 3K
150 3C 3E 3B 3D 3H 3F 3L 3K
151 3C 3J 3B 3D 3H 3F 3L 3E
152 3C 3J 3B 3D 3H 3F 3E 3K
153 3C 3E 3B 3D 3H 3F 3L 3I
154 3C 3E 3B 3D 3H 3F 3I 3K
155 3C 3J 3B 3D 3H 3F 3E 3I
156 3C 3G 3B 3D 3E 3F 3L 3K
157 3C 3G 3B 3D 3J 3F 3L 3E
158 3C 3G 3B 3D 3J 3F 3E 3K
159 3C 3G 3B 3D 3E 3F 3L 3I
160 3C 3G 3B 3D 3E 3F 3I 3K
161 3C 3G 3B 3D 3J 3F 3E 3I
162 3C 3G 3B 3D 3H 3F 3L 3E
163 3C 3G 3B 3D 3H 3F 3E 3K
164 3H 3G 3B 3C 3J 3F 3D 3E
165 3C 3G 3B 3D 3H 3F 3E 3I
166 3H 3J 3I 3F 3A 3G 3L 3K
167 3E 3J 3I 3A 3H 3G 3L 3K
168 3E 3J 3I 3F 3A 3H 3L 3K
169 3E 3J 3I 3F 3A 3G 3L 3K
170 3E 3G 3J 3F 3A 3H 3L 3K
171 3E 3G 3I 3F 3A 3H 3L 3K
172 3E 3G 3J 3F 3A 3H 3L 3I
173 3E 3G 3J 3F 3A 3H 3I 3K
174 3H 3J 3I 3D 3A 3G 3L 3K
175 3H 3J 3I 3D 3A 3F 3L 3K
176 3I 3G 3J 3D 3A 3F 3L 3K
177 3H 3G 3J 3D 3A 3F 3L 3K
178 3H 3G 3I 3D 3A 3F 3L 3K
179 3H 3G 3J 3D 3A 3F 3L 3I
180 3H 3G 3J 3D 3A 3F 3I 3K
181 3E 3J 3I 3D 3A 3H 3L 3K
182 3E 3J 3I 3D 3A 3G 3L 3K
183 3E 3G 3J 3D 3A 3H 3L 3K
184 3E 3G 3I 3D 3A 3H 3L 3K
185 3E 3G 3J 3D 3A 3H 3L 3I
186 3E 3G 3J 3D 3A 3H 3I 3K
187 3E 3J 3I 3D 3A 3F 3L 3K
188 3H 3J 3E 3D 3A 3F 3L 3K
189 3H 3E 3I 3D 3A 3F 3L 3K
190 3H 3J 3E 3D 3A 3F 3L 3I
191 3H 3J 3E 3D 3A 3F 3I 3K
192 3E 3G 3J 3D 3A 3F 3L 3K
193 3E 3G 3I 3D 3A 3F 3L 3K
194 3E 3G 3J 3D 3A 3F 3L 3I
195 3E 3G 3J 3D 3A 3F 3I 3K
196 3H 3G 3E 3D 3A 3F 3L 3K
197 3H 3G 3J 3D 3A 3F 3L 3E
198 3H 3G 3J 3D 3A 3F 3E 3K
199 3H 3G 3E 3D 3A 3F 3L 3I
200 3H 3G 3E 3D 3A 3F 3I 3K
201 3H 3G 3J 3D 3A 3F 3E 3I
202 3H 3J 3I 3C 3A 3G 3L 3K
203 3H 3J 3I 3C 3A 3F 3L 3K
204 3I 3G 3J 3C 3A 3F 3L 3K
205 3H 3G 3J 3C 3A 3F 3L 3K
206 3H 3G 3I 3C 3A 3F 3L 3K
207 3H 3G 3J 3C 3A 3F 3L 3I
208 3H 3G 3J 3C 3A 3F 3I 3K
209 3E 3J 3I 3C 3A 3H 3L 3K
210 3E 3J 3I 3C 3A 3G 3L 3K
211 3E 3G 3J 3C 3A 3H 3L 3K
212 3E 3G 3I 3C 3A 3H 3L 3K
213 3E 3G 3J 3C 3A 3H 3L 3I
214 3E 3G 3J 3C 3A 3H 3I 3K
215 3E 3J 3I 3C 3A 3F 3L 3K
216 3H 3J 3E 3C 3A 3F 3L 3K
217 3H 3E 3I 3C 3A 3F 3L 3K
218 3H 3J 3E 3C 3A 3F 3L 3I
219 3H 3J 3E 3C 3A 3F 3I 3K
220 3E 3G 3J 3C 3A 3F 3L 3K
221 3E 3G 3I 3C 3A 3F 3L 3K
222 3E 3G 3J 3C 3A 3F 3L 3I
223 3E 3G 3J 3C 3A 3F 3I 3K
224 3H 3G 3E 3C 3A 3F 3L 3K
225 3H 3G 3J 3C 3A 3F 3L 3E
226 3H 3G 3J 3C 3A 3F 3E 3K
227 3H 3G 3E 3C 3A 3F 3L 3I
228 3H 3G 3E 3C 3A 3F 3I 3K
229 3H 3G 3J 3C 3A 3F 3E 3I
230 3H 3J 3I 3C 3A 3D 3L 3K
231 3I 3G 3J 3C 3A 3D 3L 3K
232 3H 3G 3J 3C 3A 3D 3L 3K
233 3H 3G 3I 3C 3A 3D 3L 3K
234 3H 3G 3J 3C 3A 3D 3L 3I
235 3H 3G 3J 3C 3A 3D 3I 3K
236 3C 3J 3I 3D 3A 3F 3L 3K
237 3H 3J 3F 3C 3A 3D 3L 3K
238 3H 3F 3I 3C 3A 3D 3L 3K
239 3H 3J 3F 3C 3A 3D 3L 3I
240 3H 3J 3F 3C 3A 3D 3I 3K
241 3C 3G 3J 3D 3A 3F 3L 3K
242 3C 3G 3I 3D 3A 3F 3L 3K
243 3C 3G 3J 3D 3A 3F 3L 3I
244 3C 3G 3J 3D 3A 3F 3I 3K
245 3H 3G 3F 3C 3A 3D 3L 3K
246 3C 3G 3J 3D 3A 3F 3L 3H
247 3H 3G 3J 3C 3A 3F 3D 3K
248 3H 3G 3F 3C 3A 3D 3L 3I
249 3H 3G 3F 3C 3A 3D 3I 3K
250 3H 3G 3J 3C 3A 3F 3D 3I
251 3E 3J 3I 3C 3A 3D 3L 3K
252 3H 3J 3E 3C 3A 3D 3L 3K
253 3H 3E 3I 3C 3A 3D 3L 3K
254 3H 3J 3E 3C 3A 3D 3L 3I
255 3H 3J 3E 3C 3A 3D 3I 3K
256 3E 3G 3J 3C 3A 3D 3L 3K
257 3E 3G 3I 3C 3A 3D 3L 3K
258 3E 3G 3J 3C 3A 3D 3L 3I
259 3E 3G 3J 3C 3A 3D 3I 3K
260 3H 3G 3E 3C 3A 3D 3L 3K
261 3H 3G 3J 3C 3A 3D 3L 3E
262 3H 3G 3J 3C 3A 3D 3E 3K
263 3H 3G 3E 3C 3A 3D 3L 3I
264 3H 3G 3E 3C 3A 3D 3I 3K
265 3H 3G 3J 3C 3A 3D 3E 3I
266 3C 3J 3E 3D 3A 3F 3L 3K
267 3C 3E 3I 3D 3A 3F 3L 3K
268 3C 3J 3E 3D 3A 3F 3L 3I
269 3C 3J 3E 3D 3A 3F 3I 3K
270 3H 3E 3F 3C 3A 3D 3L 3K
271 3H 3J 3F 3C 3A 3D 3L 3E
272 3H 3J 3E 3C 3A 3F 3D 3K
273 3H 3E 3F 3C 3A 3D 3L 3I
274 3H 3E 3F 3C 3A 3D 3I 3K
275 3H 3J 3E 3C 3A 3F 3D 3I
276 3C 3G 3E 3D 3A 3F 3L 3K
277 3C 3G 3J 3D 3A 3F 3L 3E
278 3C 3G 3J 3D 3A 3F 3E 3K
279 3C 3G 3E 3D 3A 3F 3L 3I
280 3C 3G 3E 3D 3A 3F 3I 3K
281 3C 3G 3J 3D 3A 3F 3E 3I
282 3H 3G 3F 3C 3A 3D 3L 3E
283 3H 3G 3E 3C 3A 3F 3D 3K
284 3H 3G 3J 3C 3A 3F 3D 3E
285 3H 3G 3E 3C 3A 3F 3D 3I
286 3H 3J 3B 3A 3I 3G 3L 3K
287 3H 3J 3B 3A 3I 3F 3L 3K
288 3I 3J 3B 3F 3A 3G 3L 3K
289 3H 3J 3B 3F 3A 3G 3L 3K
290 3H 3G 3B 3A 3I 3F 3L 3K
291 3H 3J 3B 3F 3A 3G 3L 3I
292 3H 3J 3B 3F 3A 3G 3I 3K
293 3E 3J 3B 3A 3I 3H 3L 3K
294 3E 3J 3B 3A 3I 3G 3L 3K
295 3E 3J 3B 3A 3H 3G 3L 3K
296 3E 3G 3B 3A 3I 3H 3L 3K
297 3E 3J 3B 3A 3H 3G 3L 3I
298 3E 3J 3B 3A 3H 3G 3I 3K
299 3E 3J 3B 3A 3I 3F 3L 3K
300 3E 3J 3B 3F 3A 3H 3L 3K
301 3E 3I 3B 3F 3A 3H 3L 3K
302 3E 3J 3B 3F 3A 3H 3L 3I
303 3E 3J 3B 3F 3A 3H 3I 3K
304 3E 3J 3B 3F 3A 3G 3L 3K
305 3E 3G 3B 3A 3I 3F 3L 3K
306 3E 3J 3B 3F 3A 3G 3L 3I
307 3E 3J 3B 3F 3A 3G 3I 3K
308 3E 3G 3B 3F 3A 3H 3L 3K
309 3H 3J 3B 3F 3A 3G 3L 3E
310 3H 3J 3B 3F 3A 3G 3E 3K
311 3E 3G 3B 3F 3A 3H 3L 3I
312 3E 3G 3B 3F 3A 3H 3I 3K
313 3H 3J 3B 3F 3A 3G 3E 3I
314 3I 3J 3B 3D 3A 3H 3L 3K
315 3I 3J 3B 3D 3A 3G 3L 3K
316 3H 3J 3B 3D 3A 3G 3L 3K
317 3I 3G 3B 3D 3A 3H 3L 3K
318 3H 3J 3B 3D 3A 3G 3L 3I
319 3H 3J 3B 3D 3A 3G 3I 3K
320 3I 3J 3B 3D 3A 3F 3L 3K
321 3H 3J 3B 3D 3A 3F 3L 3K
322 3H 3I 3B 3D 3A 3F 3L 3K
323 3H 3J 3B 3D 3A 3F 3L 3I
324 3H 3J 3B 3D 3A 3F 3I 3K
325 3F 3J 3B 3D 3A 3G 3L 3K
326 3I 3G 3B 3D 3A 3F 3L 3K
327 3F 3J 3B 3D 3A 3G 3L 3I
328 3F 3J 3B 3D 3A 3G 3I 3K
329 3H 3G 3B 3D 3A 3F 3L 3K
330 3H 3G 3B 3D 3A 3F 3L 3J
331 3H 3G 3B 3D 3A 3F 3J 3K
332 3H 3G 3B 3D 3A 3F 3L 3I
333 3H 3G 3B 3D 3A 3F 3I 3K
334 3H 3G 3B 3D 3A 3F 3I 3J
335 3E 3J 3B 3A 3I 3D 3L 3K
336 3E 3J 3B 3D 3A 3H 3L 3K
337 3E 3I 3B 3D 3A 3H 3L 3K
338 3E 3J 3B 3D 3A 3H 3L 3I
339 3E 3J 3B 3D 3A 3H 3I 3K
340 3E 3J 3B 3D 3A 3G 3L 3K
341 3E 3G 3B 3A 3I 3D 3L 3K
342 3E 3J 3B 3D 3A 3G 3L 3I
343 3E 3J 3B 3D 3A 3G 3I 3K
344 3E 3G 3B 3D 3A 3H 3L 3K
345 3H 3J 3B 3D 3A 3G 3L 3E
346 3H 3J 3B 3D 3A 3G 3E 3K
347 3E 3G 3B 3D 3A 3H 3L 3I
348 3E 3G 3B 3D 3A 3H 3I 3K
349 3H 3J 3B 3D 3A 3G 3E 3I
350 3E 3J 3B 3D 3A 3F 3L 3K
351 3E 3I 3B 3D 3A 3F 3L 3K
352 3E 3J 3B 3D 3A 3F 3L 3I
353 3E 3J 3B 3D 3A 3F 3I 3K
354 3H 3E 3B 3D 3A 3F 3L 3K
355 3H 3J 3B 3D 3A 3F 3L 3E
356 3H 3J 3B 3D 3A 3F 3E 3K
357 3H 3E 3B 3D 3A 3F 3L 3I
358 3H 3E 3B 3D 3A 3F 3I 3K
359 3H 3J 3B 3D 3A 3F 3E 3I
360 3E 3G 3B 3D 3A 3F 3L 3K
361 3E 3G 3B 3D 3A 3F 3L 3J
362 3E 3G 3B 3D 3A 3F 3J 3K
363 3E 3G 3B 3D 3A 3F 3L 3I
364 3E 3G 3B 3D 3A 3F 3I 3K
365 3E 3G 3B 3D 3A 3F 3I 3J
366 3H 3G 3B 3D 3A 3F 3L 3E
367 3H 3G 3B 3D 3A 3F 3E 3K
368 3H 3G 3B 3D 3A 3F 3E 3J
369 3H 3G 3B 3D 3A 3F 3E 3I
370 3I 3J 3B 3C 3A 3H 3L 3K
371 3I 3J 3B 3C 3A 3G 3L 3K
372 3H 3J 3B 3C 3A 3G 3L 3K
373 3I 3G 3B 3C 3A 3H 3L 3K
374 3H 3J 3B 3C 3A 3G 3L 3I
375 3H 3J 3B 3C 3A 3G 3I 3K
376 3I 3J 3B 3C 3A 3F 3L 3K
377 3H 3J 3B 3C 3A 3F 3L 3K
378 3H 3I 3B 3C 3A 3F 3L 3K
379 3H 3J 3B 3C 3A 3F 3L 3I
380 3H 3J 3B 3C 3A 3F 3I 3K
381 3C 3J 3B 3F 3A 3G 3L 3K
382 3I 3G 3B 3C 3A 3F 3L 3K
383 3C 3J 3B 3F 3A 3G 3L 3I
384 3C 3J 3B 3F 3A 3G 3I 3K
385 3H 3G 3B 3C 3A 3F 3L 3K
386 3H 3G 3B 3C 3A 3F 3L 3J
387 3H 3G 3B 3C 3A 3F 3J 3K
388 3H 3G 3B 3C 3A 3F 3L 3I
389 3H 3G 3B 3C 3A 3F 3I 3K
390 3H 3G 3B 3C 3A 3F 3I 3J
391 3E 3J 3B 3A 3I 3C 3L 3K
392 3E 3J 3B 3C 3A 3H 3L 3K
393 3E 3I 3B 3C 3A 3H 3L 3K
394 3E 3J 3B 3C 3A 3H 3L 3I
395 3E 3J 3B 3C 3A 3H 3I 3K
396 3E 3J 3B 3C 3A 3G 3L 3K
397 3E 3G 3B 3A 3I 3C 3L 3K
398 3E 3J 3B 3C 3A 3G 3L 3I
399 3E 3J 3B 3C 3A 3G 3I 3K
400 3E 3G 3B 3C 3A 3H 3L 3K
401 3H 3J 3B 3C 3A 3G 3L 3E
402 3H 3J 3B 3C 3A 3G 3E 3K
403 3E 3G 3B 3C 3A 3H 3L 3I
404 3E 3G 3B 3C 3A 3H 3I 3K
405 3H 3J 3B 3C 3A 3G 3E 3I
406 3E 3J 3B 3C 3A 3F 3L 3K
407 3E 3I 3B 3C 3A 3F 3L 3K
408 3E 3J 3B 3C 3A 3F 3L 3I
409 3E 3J 3B 3C 3A 3F 3I 3K
410 3H 3E 3B 3C 3A 3F 3L 3K
411 3H 3J 3B 3C 3A 3F 3L 3E
412 3H 3J 3B 3C 3A 3F 3E 3K
413 3H 3E 3B 3C 3A 3F 3L 3I
414 3H 3E 3B 3C 3A 3F 3I 3K
415 3H 3J 3B 3C 3A 3F 3E 3I
416 3E 3G 3B 3C 3A 3F 3L 3K
417 3E 3G 3B 3C 3A 3F 3L 3J
418 3E 3G 3B 3C 3A 3F 3J 3K
419 3E 3G 3B 3C 3A 3F 3L 3I
420 3E 3G 3B 3C 3A 3F 3I 3K
421 3E 3G 3B 3C 3A 3F 3I 3J
422 3H 3G 3B 3C 3A 3F 3L 3E
423 3H 3G 3B 3C 3A 3F 3E 3K
424 3H 3G 3B 3C 3A 3F 3E 3J
425 3H 3G 3B 3C 3A 3F 3E 3I
426 3I 3J 3B 3C 3A 3D 3L 3K
427 3H 3J 3B 3C 3A 3D 3L 3K
428 3H 3I 3B 3C 3A 3D 3L 3K
429 3H 3J 3B 3C 3A 3D 3L 3I
430 3H 3J 3B 3C 3A 3D 3I 3K
431 3C 3J 3B 3D 3A 3G 3L 3K
432 3I 3G 3B 3C 3A 3D 3L 3K
433 3C 3J 3B 3D 3A 3G 3L 3I
434 3C 3J 3B 3D 3A 3G 3I 3K
435 3H 3G 3B 3C 3A 3D 3L 3K
436 3H 3G 3B 3C 3A 3D 3L 3J
437 3H 3G 3B 3C 3A 3D 3J 3K
438 3H 3G 3B 3C 3A 3D 3L 3I
439 3H 3G 3B 3C 3A 3D 3I 3K
440 3H 3G 3B 3C 3A 3D 3I 3J
441 3C 3J 3B 3D 3A 3F 3L 3K
442 3C 3I 3B 3D 3A 3F 3L 3K
443 3C 3J 3B 3D 3A 3F 3L 3I
444 3C 3J 3B 3D 3A 3F 3I 3K
445 3H 3F 3B 3C 3A 3D 3L 3K
446 3C 3J 3B 3D 3A 3F 3L 3H
447 3H 3J 3B 3C 3A 3F 3D 3K
448 3H 3F 3B 3C 3A 3D 3L 3I
449 3H 3F 3B 3C 3A 3D 3I 3K
450 3H 3J 3B 3C 3A 3F 3D 3I
451 3C 3G 3B 3D 3A 3F 3L 3K
452 3C 3G 3B 3D 3A 3F 3L 3J
453 3C 3G 3B 3D 3A 3F 3J 3K
454 3C 3G 3B 3D 3A 3F 3L 3I
455 3C 3G 3B 3D 3A 3F 3I 3K
456 3C 3G 3B 3D 3A 3F 3I 3J
457 3C 3G 3B 3D 3A 3F 3L 3H
458 3H 3G 3B 3C 3A 3F 3D 3K
459 3H 3G 3B 3C 3A 3F 3D 3J
460 3H 3G 3B 3C 3A 3F 3D 3I
461 3E 3J 3B 3C 3A 3D 3L 3K
462 3E 3I 3B 3C 3A 3D 3L 3K
463 3E 3J 3B 3C 3A 3D 3L 3I
464 3E 3J 3B 3C 3A 3D 3I 3K
465 3H 3E 3B 3C 3A 3D 3L 3K
466 3H 3J 3B 3C 3A 3D 3L 3E
467 3H 3J 3B 3C 3A 3D 3E 3K
468 3H 3E 3B 3C 3A 3D 3L 3I
469 3H 3E 3B 3C 3A 3D 3I 3K
470 3H 3J 3B 3C 3A 3D 3E 3I
471 3E 3G 3B 3C 3A 3D 3L 3K
472 3E 3G 3B 3C 3A 3D 3L 3J
473 3E 3G 3B 3C 3A 3D 3J 3K
474 3E 3G 3B 3C 3A 3D 3L 3I
475 3E 3G 3B 3C 3A 3D 3I 3K
476 3E 3G 3B 3C 3A 3D 3I 3J
477 3H 3G 3B 3C 3A 3D 3L 3E
478 3H 3G 3B 3C 3A 3D 3E 3K
479 3H 3G 3B 3C 3A 3D 3E 3J
480 3H 3G 3B 3C 3A 3D 3E 3I
481 3C 3E 3B 3D 3A 3F 3L 3K
482 3C 3J 3B 3D 3A 3F 3L 3E
483 3C 3J 3B 3D 3A 3F 3E 3K
484 3C 3E 3B 3D 3A 3F 3L 3I
485 3C 3E 3B 3D 3A 3F 3I 3K
486 3C 3J 3B 3D 3A 3F 3E 3I
487 3H 3F 3B 3C 3A 3D 3L 3E
488 3H 3E 3B 3C 3A 3F 3D 3K
489 3H 3J 3B 3C 3A 3F 3D 3E
490 3H 3E 3B 3C 3A 3F 3D 3I
491 3C 3G 3B 3D 3A 3F 3L 3E
492 3C 3G 3B 3D 3A 3F 3E 3K
493 3C 3G 3B 3D 3A 3F 3E 3J
494 3C 3G 3B 3D 3A 3F 3E 3I
495 3H 3G 3B 3C 3A 3F 3D 3E"""

# Parse into lookup dict: frozenset of 8 group letters -> slot assignments
SLOTS = ['1A', '1B', '1D', '1E', '1G', '1I', '1K', '1L']

annexe_c = {}
for line in ANNEXE_C_RAW.strip().split('\n'):
    parts = line.split()
    groups = [p.replace('3', '') for p in parts[1:]]  # ['E','J','I','F','H','G','L','K']
    key = frozenset(groups)
    annexe_c[key] = {slot: grp for slot, grp in zip(SLOTS, groups)}

print(f"Annexe C loaded: {len(annexe_c)} combinations")

Annexe C loaded: 495 combinations


In [32]:
def get_third_place_teams(all_group_results):
    """
    all_group_results: dict of group_letter -> ranked list of 4 team dicts
    Returns: list of 8 best 3rd place team dicts, sorted best to worst
    """
    third_place = []
    for group, ranked in all_group_results.items():
        t = ranked[2].copy()  # 3rd place team
        t['group'] = group
        third_place.append(t)
    
    # Rank by: pts → GD → GF → synthetic Elo (no H2H for 3rd place ranking)
    third_place.sort(key=lambda t: (
        t['pts'],
        t['gd'],
        t['gf'],
        team_elo.get(t['team'], 1500)
    ), reverse=True)
    
    return third_place[:8]


def get_annexe_c_assignments(best_thirds):
    """
    Given the 8 best 3rd place teams, look up Annexe C to get R32 slot assignments.
    Returns dict: slot -> team name
    e.g. {'1A': 'Norway', '1B': 'Ecuador', ...}
    """
    group_letters = frozenset([t['group'] for t in best_thirds])
    
    if group_letters not in annexe_c:
        raise ValueError(f"Combination {group_letters} not found in Annexe C")
    
    slot_to_group = annexe_c[group_letters]
    
    # Map group letter -> team name
    group_to_team = {t['group']: t['team'] for t in best_thirds}
    
    slot_to_team = {slot: group_to_team[grp] for slot, grp in slot_to_group.items()}
    return slot_to_team

In [33]:
# Simulate all 12 groups
all_group_results = {}
for group, teams in GROUPS.items():
    ranked, results = simulate_group(group, teams)
    all_group_results[group] = ranked

# Print group winners and 3rd place
for group, ranked in all_group_results.items():
    print(f"Group {group}: 1st={ranked[0]['team']} 2nd={ranked[1]['team']} 3rd={ranked[2]['team']} 4th={ranked[3]['team']}")

print()
best_thirds = get_third_place_teams(all_group_results)
print("Best 8 third place teams:")
for t in best_thirds:
    print(f"  Group {t['group']}: {t['team']} ({t['pts']}pts, GD={t['gd']})")

print()
assignments = get_annexe_c_assignments(best_thirds)
print("R32 slot assignments:")
for slot, team in assignments.items():
    print(f"  {slot} → {team}")

Group A: 1st=Czech Republic 2nd=South Africa 3rd=Mexico 4th=South Korea
Group B: 1st=Bosnia and Herzegovina 2nd=Qatar 3rd=Canada 4th=Switzerland
Group C: 1st=Brazil 2nd=Morocco 3rd=Scotland 4th=Haiti
Group D: 1st=Turkey 2nd=United States 3rd=Australia 4th=Paraguay
Group E: 1st=Germany 2nd=Curaçao 3rd=Ecuador 4th=Ivory Coast
Group F: 1st=Netherlands 2nd=Sweden 3rd=Tunisia 4th=Japan
Group G: 1st=Iran 2nd=New Zealand 3rd=Belgium 4th=Egypt
Group H: 1st=Spain 2nd=Uruguay 3rd=Saudi Arabia 4th=Cape Verde
Group I: 1st=France 2nd=Norway 3rd=Senegal 4th=Iraq
Group J: 1st=Austria 2nd=Argentina 3rd=Algeria 4th=Jordan
Group K: 1st=Portugal 2nd=Colombia 3rd=DR Congo 4th=Uzbekistan
Group L: 1st=Croatia 2nd=Panama 3rd=England 4th=Ghana

Best 8 third place teams:
  Group A: Mexico (4pts, GD=1)
  Group L: England (4pts, GD=-1)
  Group J: Algeria (4pts, GD=-1)
  Group B: Canada (4pts, GD=-1)
  Group F: Tunisia (3pts, GD=-3)
  Group H: Saudi Arabia (3pts, GD=-5)
  Group I: Senegal (3pts, GD=-5)
  Group G:

In [34]:
def build_r32_bracket(all_group_results, slot_to_team):
    """
    Wire up all 16 R32 matches per Article 12.6.
    Returns list of 16 (team_a, team_b) tuples.
    """
    # Group winners and runners-up
    w = {g: all_group_results[g][0]['team'] for g in 'ABCDEFGHIJKL'}
    ru = {g: all_group_results[g][1]['team'] for g in 'ABCDEFGHIJKL'}
    t = slot_to_team  # 3rd place slot assignments

    r32 = [
        (ru['A'],  ru['B']),      # M73
        (w['E'],   t['1E']),      # M74
        (w['F'],   ru['C']),      # M75
        (w['C'],   ru['F']),      # M76
        (w['I'],   t['1I']),      # M77
        (ru['E'],  ru['I']),      # M78
        (w['A'],   t['1A']),      # M79
        (w['L'],   t['1L']),      # M80
        (w['D'],   t['1D']),      # M81
        (w['G'],   t['1G']),      # M82
        (ru['K'],  ru['L']),      # M83
        (w['H'],   ru['J']),      # M84
        (w['B'],   t['1B']),      # M85
        (w['J'],   ru['H']),      # M86
        (w['K'],   t['1K']),      # M87
        (ru['D'],  ru['G']),      # M88
    ]
    return r32

In [35]:
best_thirds = get_third_place_teams(all_group_results)
slot_to_team = get_annexe_c_assignments(best_thirds)
r32 = build_r32_bracket(all_group_results, slot_to_team)

print("R32 matches:")
for i, (a, b) in enumerate(r32):
    print(f"  M{73+i}: {a} vs {b}")
    

R32 matches:
  M73: South Africa vs Qatar
  M74: Germany vs Tunisia
  M75: Netherlands vs Morocco
  M76: Brazil vs Sweden
  M77: France vs Belgium
  M78: Curaçao vs Norway
  M79: Czech Republic vs Saudi Arabia
  M80: Croatia vs Senegal
  M81: Turkey vs Canada
  M82: Iran vs Mexico
  M83: Colombia vs Panama
  M84: Spain vs Argentina
  M85: Bosnia and Herzegovina vs Algeria
  M86: Austria vs Uruguay
  M87: Portugal vs England
  M88: United States vs New Zealand


In [36]:
def simulate_knockout_round(matches):
    """
    Given a list of (team_a, team_b) tuples, simulate each match.
    Returns list of winners in same order.
    """
    winners = []
    for team_a, team_b in matches:
        winner, hg, ag, method = simulate_knockout_match(team_a, team_b)
        winners.append(winner)
    return winners


def simulate_knockout_stage(r32_matches):
    """
    Simulate R32 → R16 → QF → SF → Final per Article 12.7-12.11.
    Returns (winner, round_reached dict)
    """
    round_reached = {}

    # R32 — 16 matches, winners go to R16
    r32_winners = simulate_knockout_round(r32_matches)
    for team_a, team_b in r32_matches:
        for team in [team_a, team_b]:
            round_reached[team] = 'round_of_32'

    # R16 — Article 12.7 bracket structure
    # Pairs of R32 winners: (M74,M77), (M73,M75), (M76,M78), (M79,M80),
    #                       (M83,M84), (M81,M82), (M86,M88), (M85,M87)
    r16_matches = [
        (r32_winners[1],  r32_winners[4]),   # M89: W74 vs W77
        (r32_winners[0],  r32_winners[2]),   # M90: W73 vs W75
        (r32_winners[3],  r32_winners[5]),   # M91: W76 vs W78
        (r32_winners[6],  r32_winners[7]),   # M92: W79 vs W80
        (r32_winners[10], r32_winners[11]),  # M93: W83 vs W84
        (r32_winners[8],  r32_winners[9]),   # M94: W81 vs W82
        (r32_winners[13], r32_winners[15]),  # M95: W86 vs W88
        (r32_winners[12], r32_winners[14]),  # M96: W85 vs W87
    ]

    r16_winners = simulate_knockout_round(r16_matches)
    for team in r32_winners:
        round_reached[team] = 'round_of_16'

    # QF — Article 12.8
    qf_matches = [
        (r16_winners[0], r16_winners[1]),  # M97: W89 vs W90
        (r16_winners[4], r16_winners[5]),  # M98: W93 vs W94
        (r16_winners[2], r16_winners[3]),  # M99: W91 vs W92
        (r16_winners[6], r16_winners[7]),  # M100: W95 vs W96
    ]

    qf_winners = simulate_knockout_round(qf_matches)
    for team in r16_winners:
        round_reached[team] = 'quarter_final'

    # SF — Article 12.9
    sf_matches = [
        (qf_winners[0], qf_winners[1]),  # M101: W97 vs W98
        (qf_winners[2], qf_winners[3]),  # M102: W99 vs W100
    ]

    sf_winners = simulate_knockout_round(sf_matches)
    for team in qf_winners:
        round_reached[team] = 'semi_final'

    # Final — Article 12.11
    finalist_a, finalist_b = sf_winners[0], sf_winners[1]
    final_winner, hg, ag, method = simulate_knockout_match(finalist_a, finalist_b)
    for team in sf_winners:
        round_reached[team] = 'final'

    round_reached[final_winner] = 'win'

    return final_winner, round_reached

In [37]:
winner, round_reached = simulate_knockout_stage(r32)

print(f"Winner: {winner}")
print("\nRound reached per team:")
rounds_order = ['round_of_32', 'round_of_16', 'quarter_final', 'semi_final', 'final', 'win']
for round_name in rounds_order:
    teams = [t for t, r in round_reached.items() if r == round_name]
    print(f"  {round_name}: {', '.join(teams)}")

Winner: Germany

Round reached per team:
  round_of_32: Qatar, Tunisia, Netherlands, Sweden, Belgium, Curaçao, Saudi Arabia, Senegal, Canada, Mexico, Panama, Argentina, Bosnia and Herzegovina, Uruguay, England, New Zealand
  round_of_16: South Africa, Brazil, France, Czech Republic, Iran, Colombia, Algeria, Austria
  quarter_final: Morocco, Norway, Spain, United States
  semi_final: Croatia, Turkey
  final: Portugal
  win: Germany


In [38]:
def simulate_tournament():
    """
    Simulate one complete WC 2026.
    Returns (winner, round_reached dict, all_group_results)
    """
    # Step 1 — Simulate all 12 groups
    all_group_results = {}
    all_match_results = {}
    for group, teams in GROUPS.items():
        ranked, match_results = simulate_group(group, teams)
        all_group_results[group] = ranked
        all_match_results[group] = match_results

    # Step 2 — Get qualified teams
    # Group winners and runners-up (24 teams)
    # 8 best 3rd place teams
    best_thirds = get_third_place_teams(all_group_results)
    slot_to_team = get_annexe_c_assignments(best_thirds)

    # Step 3 — Build R32 bracket
    r32_matches = build_r32_bracket(all_group_results, slot_to_team)

    # Step 4 — Simulate knockout stage
    winner, round_reached = simulate_knockout_stage(r32_matches)

    # Step 5 — Add group stage exits for eliminated teams
    for group, ranked in all_group_results.items():
        # 4th place always exits at group stage
        fourth = ranked[3]['team']
        round_reached[fourth] = 'group_exit'
        # 3rd place — either qualified or group exit
        third = ranked[2]['team']
        if third not in round_reached:
            round_reached[third] = 'group_exit'

    return winner, round_reached, all_group_results

In [39]:
winner, round_reached, group_results = simulate_tournament()

print(f"Winner: {winner}")
print(f"Total teams tracked: {len(round_reached)}")

rounds_order = ['group_exit', 'round_of_32', 'round_of_16', 'quarter_final', 'semi_final', 'final', 'win']
for r in rounds_order:
    teams = [t for t, rd in round_reached.items() if rd == r]
    print(f"  {r} ({len(teams)}): {', '.join(teams)}")

Winner: Brazil
Total teams tracked: 48
  group_exit (16): South Korea, South Africa, Canada, Bosnia and Herzegovina, Haiti, Australia, Ecuador, Tunisia, New Zealand, Saudi Arabia, Iraq, Norway, Jordan, DR Congo, Panama, Ghana
  round_of_32 (16): Switzerland, Paraguay, Morocco, Sweden, Japan, Senegal, Scotland, Uzbekistan, Turkey, Cape Verde, Colombia, Austria, Iran, Uruguay, Curaçao, Egypt
  round_of_16 (8): Mexico, Ivory Coast, Germany, Croatia, Argentina, Spain, Qatar, Algeria
  quarter_final (4): France, Czech Republic, England, Portugal
  semi_final (2): Belgium, United States
  final (1): Netherlands
  win (1): Brazil


In [44]:
N_SIMULATIONS = 10000

# Accumulators
round_counts = defaultdict(lambda: defaultdict(int))
goals_scored = defaultdict(list)
goals_conceded = defaultdict(list)
win_counts = defaultdict(int)

print(f"Running {N_SIMULATIONS} simulations...")

for i in range(N_SIMULATIONS):
    winner, round_reached, group_results = simulate_tournament()
    
    # Count rounds
    for team, rd in round_reached.items():
        round_counts[team][rd] += 1
    
    # Count wins
    win_counts[winner] += 1
    
    # Track avg goals from group stage
    for group, ranked in group_results.items():
        for team_row in ranked:
            team = team_row['team']
            goals_scored[team].append(team_row['gf'])
            goals_conceded[team].append(team_row['ga'])

    # Print progress every 500
    if (i + 1) % 500 == 0:
        print(f"\n--- {i+1}/{N_SIMULATIONS} simulations ---")
        top5 = sorted(win_counts.items(), key=lambda x: x[1], reverse=True)[:5]
        for team, count in top5:
            print(f"  {team}: {count/(i+1)*100:.1f}%")

print("\nDone!")
print(f"\nFinal top 10:")
for team, count in sorted(win_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {team}: {count/N_SIMULATIONS*100:.1f}%")

Running 10000 simulations...


KeyboardInterrupt: 

In [45]:
print(penalty_lookup.get('Austria', 'NOT FOUND'))
print(penalty_lookup.get('Curaçao', 'NOT FOUND'))
print(penalty_lookup.get('Jordan', 'NOT FOUND'))

0.1
0.1
0.1


In [49]:
def predict_lambdas(home_team, away_team, is_wc=1):
    return lambda_cache[(home_team, away_team)]

In [50]:
import inspect
print(inspect.getsource(predict_lambdas))

def predict_lambdas(home_team, away_team, is_wc=1):
    return lambda_cache[(home_team, away_team)]



In [52]:
import time
start = time.time()

all_teams = [team for teams in GROUPS.values() for team in teams]

lambda_cache = {}
for home in all_teams:
    for away in all_teams:
        if home == away:
            continue
        h_elo = team_elo[home]
        a_elo = team_elo[away]
        features = pd.DataFrame([{
            "home_elo": h_elo,
            "away_elo": a_elo,
            "elo_diff": h_elo - a_elo,
            "is_worldcup": 1
        }])
        lam_h = float(np.clip(home_model.predict(features)[0], 0.1, 2.5))
        lam_a = float(np.clip(away_model.predict(features)[0], 0.1, 2.5))
        lambda_cache[(home, away)] = (lam_h, lam_a)

print(f"Precomputed {len(lambda_cache)} lambdas in {time.time()-start:.2f}s")

Precomputed 2256 lambdas in 9.47s


In [54]:
start = time.time()
for i in range(1):
    simulate_tournament()
elapsed = time.time() - start
print(f"100 simulations in {elapsed:.1f}s")
print(f"Estimated time for 10k: {elapsed * 100 / 60:.1f} minutes")

100 simulations in 1.9s
Estimated time for 10k: 3.2 minutes


In [56]:
import time
start = time.time()

grid_cache = {}
for home in all_teams:
    for away in all_teams:
        if home == away:
            continue
        lam_h, lam_a = lambda_cache[(home, away)]
        grid_cache[(home, away)] = scoreline_probs(lam_h, lam_a, RHO)

print(f"Built {len(grid_cache)} grids in {time.time()-start:.2f}s")

Built 2256 grids in 34.82s


In [57]:
flat_cache = {}
for home in all_teams:
    for away in all_teams:
        if home == away:
            continue
        grid = grid_cache[(home, away)]
        flat = grid.flatten()
        flat_cache[(home, away)] = (flat, np.cumsum(flat))

print(f"Built {len(flat_cache)} flat arrays")

Built 2256 flat arrays


In [58]:
def simulate_group_match(home_team, away_team):
    flat, cumsum = flat_cache[(home_team, away_team)]
    r = np.random.random()
    idx = min(np.searchsorted(cumsum, r), len(flat) - 1)
    h = idx // 9
    a = idx % 9
    return h, a

def simulate_knockout_match(team_a, team_b):
    flat, cumsum = flat_cache[(team_a, team_b)]
    r = np.random.random()
    idx = min(np.searchsorted(cumsum, r), len(flat) - 1)
    h = idx // 9
    a = idx % 9

    if h != a:
        winner = team_a if h > a else team_b
        return (winner, h, a, 'normal')

    lam_a, lam_b = lambda_cache[(team_a, team_b)]
    et_grid = scoreline_probs(lam_a * 0.33, lam_b * 0.33, RHO)
    et_flat = et_grid.flatten()
    et_cumsum = np.cumsum(et_flat)
    r = np.random.random()
    et_idx = min(np.searchsorted(et_cumsum, r), len(et_flat) - 1)
    et_h = et_idx // 9
    et_a = et_idx % 9
    h += et_h
    a += et_a

    if h != a:
        winner = team_a if h > a else team_b
        return (winner, h, a, 'et')

    winner = simulate_penalty(team_a, team_b)
    return (winner, h, a, 'pens')

In [59]:
import time
start = time.time()
simulate_tournament()
print(f"1 simulation: {time.time()-start:.3f}s")

1 simulation: 0.101s


In [60]:
from collections import defaultdict
import time

N_SIMULATIONS = 10000

round_counts = defaultdict(lambda: defaultdict(int))
goals_scored = defaultdict(list)
goals_conceded = defaultdict(list)
win_counts = defaultdict(int)

start = time.time()
print(f"Running {N_SIMULATIONS} simulations...")

for i in range(N_SIMULATIONS):
    winner, round_reached, group_results = simulate_tournament()
    
    for team, rd in round_reached.items():
        round_counts[team][rd] += 1
    
    win_counts[winner] += 1
    
    for group, ranked in group_results.items():
        for team_row in ranked:
            team = team_row['team']
            goals_scored[team].append(team_row['gf'])
            goals_conceded[team].append(team_row['ga'])

    if (i + 1) % 500 == 0:
        elapsed = time.time() - start
        remaining = (elapsed / (i+1)) * (N_SIMULATIONS - i - 1)
        print(f"\n--- {i+1}/{N_SIMULATIONS} | {elapsed/60:.1f}m elapsed | ~{remaining/60:.1f}m remaining ---")
        top5 = sorted(win_counts.items(), key=lambda x: x[1], reverse=True)[:5]
        for team, count in top5:
            print(f"  {team}: {count/(i+1)*100:.1f}%")

print(f"\nDone in {(time.time()-start)/60:.1f} minutes!")
print(f"\nFinal top 10:")
for team, count in sorted(win_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {team}: {count/N_SIMULATIONS*100:.1f}%")

Running 10000 simulations...

--- 500/10000 | 1.1m elapsed | ~21.3m remaining ---
  France: 19.2%
  Spain: 19.0%
  Germany: 16.0%
  Brazil: 12.4%
  Portugal: 9.0%

--- 1000/10000 | 2.3m elapsed | ~20.5m remaining ---
  Spain: 19.3%
  France: 18.5%
  Germany: 15.7%
  Brazil: 12.1%
  Portugal: 8.1%

--- 1500/10000 | 3.4m elapsed | ~19.2m remaining ---
  Spain: 19.2%
  France: 17.7%
  Germany: 15.7%
  Brazil: 12.3%
  Portugal: 8.5%

--- 2000/10000 | 4.5m elapsed | ~18.1m remaining ---
  Spain: 19.0%
  France: 17.3%
  Germany: 15.3%
  Brazil: 11.8%
  Portugal: 8.9%

--- 2500/10000 | 5.7m elapsed | ~17.1m remaining ---
  Spain: 18.6%
  France: 17.3%
  Germany: 15.1%
  Brazil: 11.2%
  Portugal: 9.2%

--- 3000/10000 | 6.8m elapsed | ~15.9m remaining ---
  Spain: 18.8%
  France: 16.8%
  Germany: 15.0%
  Brazil: 11.0%
  Portugal: 9.5%

--- 3500/10000 | 7.9m elapsed | ~14.7m remaining ---
  Spain: 18.0%
  France: 17.0%
  Germany: 15.3%
  Brazil: 11.2%
  Portugal: 9.0%

--- 4000/10000 | 9.0m elap

In [61]:
import pickle

checkpoint = {
    'win_counts': dict(win_counts),
    'round_counts': {team: dict(rounds) for team, rounds in round_counts.items()},
    'goals_scored': dict(goals_scored),
    'goals_conceded': dict(goals_conceded),
    'n_simulations': N_SIMULATIONS,
}

with open('../data/processed/simulation_raw.pkl', 'wb') as f:
    pickle.dump(checkpoint, f)

print("Saved!")

Saved!


In [62]:
import pickle
import json
from datetime import datetime, timezone

with open('../data/processed/simulation_raw.pkl', 'rb') as f:
    checkpoint = pickle.load(f)

win_counts = checkpoint['win_counts']
round_counts = checkpoint['round_counts']
goals_scored = checkpoint['goals_scored']
goals_conceded = checkpoint['goals_conceded']
N = checkpoint['n_simulations']

print(f"Loaded {N} simulations")
print(f"Teams tracked: {len(round_counts)}")

Loaded 10000 simulations
Teams tracked: 48


In [63]:
# Confederation mapping
CONFEDERATION = {
    'Mexico': 'CONCACAF', 'South Korea': 'AFC', 'Czech Republic': 'UEFA', 'South Africa': 'CAF',
    'Canada': 'CONCACAF', 'Switzerland': 'UEFA', 'Qatar': 'AFC', 'Bosnia and Herzegovina': 'UEFA',
    'Brazil': 'CONMEBOL', 'Morocco': 'CAF', 'Scotland': 'UEFA', 'Haiti': 'CONCACAF',
    'United States': 'CONCACAF', 'Australia': 'AFC', 'Turkey': 'UEFA', 'Paraguay': 'CONMEBOL',
    'Germany': 'UEFA', 'Ivory Coast': 'CAF', 'Ecuador': 'CONMEBOL', 'Curaçao': 'CONCACAF',
    'Netherlands': 'UEFA', 'Japan': 'AFC', 'Sweden': 'UEFA', 'Tunisia': 'CAF',
    'Belgium': 'UEFA', 'Egypt': 'CAF', 'Iran': 'AFC', 'New Zealand': 'OFC',
    'Spain': 'UEFA', 'Uruguay': 'CONMEBOL', 'Saudi Arabia': 'AFC', 'Cape Verde': 'CAF',
    'France': 'UEFA', 'Senegal': 'CAF', 'Norway': 'UEFA', 'Iraq': 'AFC',
    'Argentina': 'CONMEBOL', 'Austria': 'UEFA', 'Algeria': 'CAF', 'Jordan': 'AFC',
    'Portugal': 'UEFA', 'DR Congo': 'CAF', 'Uzbekistan': 'AFC', 'Colombia': 'CONMEBOL',
    'England': 'UEFA', 'Croatia': 'UEFA', 'Ghana': 'CAF', 'Panama': 'CONCACAF',
}

# Flag mapping
FLAGS = {
    'Mexico': '🇲🇽', 'South Korea': '🇰🇷', 'Czech Republic': '🇨🇿', 'South Africa': '🇿🇦',
    'Canada': '🇨🇦', 'Switzerland': '🇨🇭', 'Qatar': '🇶🇦', 'Bosnia and Herzegovina': '🇧🇦',
    'Brazil': '🇧🇷', 'Morocco': '🇲🇦', 'Scotland': '🏴󠁧󠁢󠁳󠁣󠁴󠁿', 'Haiti': '🇭🇹',
    'United States': '🇺🇸', 'Australia': '🇦🇺', 'Turkey': '🇹🇷', 'Paraguay': '🇵🇾',
    'Germany': '🇩🇪', 'Ivory Coast': '🇨🇮', 'Ecuador': '🇪🇨', 'Curaçao': '🇨🇼',
    'Netherlands': '🇳🇱', 'Japan': '🇯🇵', 'Sweden': '🇸🇪', 'Tunisia': '🇹🇳',
    'Belgium': '🇧🇪', 'Egypt': '🇪🇬', 'Iran': '🇮🇷', 'New Zealand': '🇳🇿',
    'Spain': '🇪🇸', 'Uruguay': '🇺🇾', 'Saudi Arabia': '🇸🇦', 'Cape Verde': '🇨🇻',
    'France': '🇫🇷', 'Senegal': '🇸🇳', 'Norway': '🇳🇴', 'Iraq': '🇮🇶',
    'Argentina': '🇦🇷', 'Austria': '🇦🇹', 'Algeria': '🇩🇿', 'Jordan': '🇯🇴',
    'Portugal': '🇵🇹', 'DR Congo': '🇨🇩', 'Uzbekistan': '🇺🇿', 'Colombia': '🇨🇴',
    'England': '🏴󠁧󠁢󠁥󠁮󠁧󠁿', 'Croatia': '🇭🇷', 'Ghana': '🇬🇭', 'Panama': '🇵🇦',
}

# Group mapping
TEAM_GROUP = {team: group for group, teams in GROUPS.items() for team in teams}

def make_slug(name):
    return name.lower().replace(' ', '-').replace('.', '').replace('ç', 'c').replace('ü', 'u').replace('ã', 'a')

def get_prob(team, round_name):
    rc = round_counts.get(team, {})
    rounds_above = {
        'round_of_32': ['round_of_32', 'round_of_16', 'quarter_final', 'semi_final', 'final', 'win'],
        'round_of_16': ['round_of_16', 'quarter_final', 'semi_final', 'final', 'win'],
        'quarter_final': ['quarter_final', 'semi_final', 'final', 'win'],
        'semi_final': ['semi_final', 'final', 'win'],
        'final': ['final', 'win'],
        'win': ['win'],
    }
    count = sum(rc.get(r, 0) for r in rounds_above[round_name])
    return round(count / N, 4)

teams_output = []
for team in sorted(TEAM_GROUP.keys()):
    rc = round_counts.get(team, {})
    group_exit_count = rc.get('group_exit', 0)
    qualify_prob = round(1 - group_exit_count / N, 4)

    gs = goals_scored.get(team, [0])
    gc = goals_conceded.get(team, [0])

    teams_output.append({
        'name': team,
        'slug': make_slug(team),
        'confederation': CONFEDERATION[team],
        'flag': FLAGS[team],
        'group': TEAM_GROUP[team],
        'qualify_probability': qualify_prob,
        'probabilities': {
            'win': get_prob(team, 'win'),
            'final': get_prob(team, 'final'),
            'semi_final': get_prob(team, 'semi_final'),
            'quarter_final': get_prob(team, 'quarter_final'),
            'round_of_16': get_prob(team, 'round_of_16'),
            'round_of_32': get_prob(team, 'round_of_32'),
            'group_exit': round(group_exit_count / N, 4),
        },
        'avg_goals_scored': round(float(np.mean(gs)), 2),
        'avg_goals_conceded': round(float(np.mean(gc)), 2),
    })

# Sort by win probability
teams_output.sort(key=lambda x: x['probabilities']['win'], reverse=True)
print(f"Built {len(teams_output)} team entries")
print("Top 5:")
for t in teams_output[:5]:
    print(f"  {t['flag']} {t['name']}: {t['probabilities']['win']*100:.1f}%")

Built 48 team entries
Top 5:
  🇪🇸 Spain: 17.8%
  🇫🇷 France: 17.1%
  🇩🇪 Germany: 15.4%
  🇧🇷 Brazil: 11.6%
  🏴󠁧󠁢󠁥󠁮󠁧󠁿 England: 8.8%


In [65]:
def get_most_likely_winner(team_a, team_b):
    """Return the team with higher win probability and their win prob."""
    lam_h, lam_a = lambda_cache[(team_a, team_b)]
    grid = scoreline_probs(lam_h, lam_a, RHO)
    home_win = sum(grid[h, a] for h in range(9) for a in range(9) if h > a)
    away_win = sum(grid[h, a] for h in range(9) for a in range(9) if a > h)
    draw = 1 - home_win - away_win
    # In knockout, draw goes to ET/pens — split evenly
    prob_a = home_win + draw * 0.5
    if prob_a >= 0.5:
        return team_a, round(prob_a, 4)
    else:
        return team_b, round(1 - prob_a, 4)

# Run one deterministic-ish simulation to get bracket structure
# Use the most common group results across simulations
# Simplest approach: run one simulation and use that bracket
np.random.seed(42)
_, _, sample_group_results = simulate_tournament()

best_thirds = get_third_place_teams(sample_group_results)
slot_to_team = get_annexe_c_assignments(best_thirds)
r32 = build_r32_bracket(sample_group_results, slot_to_team)

# Build bracket dict
def build_bracket(r32_matches):
    bracket = {
        'round_of_32': [],
        'round_of_16': [],
        'quarter_finals': [],
        'semi_finals': [],
        'final': None,
        'winner': None,
    }

    # R32
    r32_winners = []
    for a, b in r32_matches:
        winner, prob = get_most_likely_winner(a, b)
        bracket['round_of_32'].append({
            'team_a': a, 'team_b': b,
            'predicted_winner': winner,
            'win_probability': prob
        })
        r32_winners.append(winner)

    # R16 per Article 12.7
    r16_matches = [
        (r32_winners[1],  r32_winners[4]),
        (r32_winners[0],  r32_winners[2]),
        (r32_winners[3],  r32_winners[5]),
        (r32_winners[6],  r32_winners[7]),
        (r32_winners[10], r32_winners[11]),
        (r32_winners[8],  r32_winners[9]),
        (r32_winners[13], r32_winners[15]),
        (r32_winners[12], r32_winners[14]),
    ]
    r16_winners = []
    for a, b in r16_matches:
        winner, prob = get_most_likely_winner(a, b)
        bracket['round_of_16'].append({
            'team_a': a, 'team_b': b,
            'predicted_winner': winner,
            'win_probability': prob
        })
        r16_winners.append(winner)

    # QF
    qf_matches = [
        (r16_winners[0], r16_winners[1]),
        (r16_winners[4], r16_winners[5]),
        (r16_winners[2], r16_winners[3]),
        (r16_winners[6], r16_winners[7]),
    ]
    qf_winners = []
    for a, b in qf_matches:
        winner, prob = get_most_likely_winner(a, b)
        bracket['quarter_finals'].append({
            'team_a': a, 'team_b': b,
            'predicted_winner': winner,
            'win_probability': prob
        })
        qf_winners.append(winner)

    # SF
    sf_matches = [
        (qf_winners[0], qf_winners[1]),
        (qf_winners[2], qf_winners[3]),
    ]
    sf_winners = []
    for a, b in sf_matches:
        winner, prob = get_most_likely_winner(a, b)
        bracket['semi_finals'].append({
            'team_a': a, 'team_b': b,
            'predicted_winner': winner,
            'win_probability': prob
        })
        sf_winners.append(winner)

    # Final
    finalist_a, finalist_b = sf_winners[0], sf_winners[1]
    winner, prob = get_most_likely_winner(finalist_a, finalist_b)
    bracket['final'] = {
        'team_a': finalist_a, 'team_b': finalist_b,
        'predicted_winner': winner,
        'win_probability': prob
    }
    bracket['winner'] = winner

    return bracket

bracket = build_bracket(r32)
print(f"Bracket winner: {bracket['winner']}")
print(f"Finalists: {bracket['final']['team_a']} vs {bracket['final']['team_b']}")

Bracket winner: France
Finalists: France vs Spain


In [66]:
output = {
    'simulated_at': datetime.now(timezone.utc).isoformat(),
    'n_simulations': N,
    'teams': teams_output,
    'bracket': bracket,
}

with open('../data/processed/simulation_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"Written simulation_results.json")
print(f"Teams: {len(output['teams'])}")
print(f"Winner: {output['bracket']['winner']}")
print(f"Top 5:")
for t in output['teams'][:5]:
    print(f"  {t['flag']} {t['name']}: {t['probabilities']['win']*100:.1f}%")

Written simulation_results.json
Teams: 48
Winner: France
Top 5:
  🇪🇸 Spain: 17.8%
  🇫🇷 France: 17.1%
  🇩🇪 Germany: 15.4%
  🇧🇷 Brazil: 11.6%
  🏴󠁧󠁢󠁥󠁮󠁧󠁿 England: 8.8%


In [67]:
with open('../data/processed/simulation_results.json') as f:
    data = json.load(f)

teams = data['teams']
names = {t['name'] for t in teams}
errors = []

# 1. Right number of teams
if len(teams) != 48:
    errors.append(f"Expected 48 teams, got {len(teams)}")

# 2. Probability sanity
for t in teams:
    p = t['probabilities']
    if p['win'] > p['final']:
        errors.append(f"{t['name']}: win prob > final prob")
    if p['group_exit'] + p['round_of_32'] > 1.001:
        errors.append(f"{t['name']}: group_exit + round_of_32 > 1")

# 3. Bracket names match team names
all_matches = (
    data['bracket']['round_of_32'] +
    data['bracket']['round_of_16'] +
    data['bracket']['quarter_finals'] +
    data['bracket']['semi_finals'] +
    [data['bracket']['final']]
)
for m in all_matches:
    for key in ['team_a', 'team_b', 'predicted_winner']:
        if m[key] not in names:
            errors.append(f"Bracket name '{m[key]}' not found in teams")

# 4. Winner in teams
if data['bracket']['winner'] not in names:
    errors.append(f"Winner '{data['bracket']['winner']}' not in teams")

# 5. Slugs unique
slugs = [t['slug'] for t in teams]
if len(slugs) != len(set(slugs)):
    errors.append("Duplicate slugs found")

if errors:
    print("VALIDATION FAILED:")
    for e in errors:
        print(f"  ✗ {e}")
else:
    print("✓ Validation passed — safe to copy to frontend")

✓ Validation passed — safe to copy to frontend
